### Structure

In [5]:
import os

In [ ]:
project_folder = r"D:/GitHub/GML_Project"
os.chdir(project_folder)

try:
    os.chdir(r"./Pictures/Tierbilder_Dataset_Train_Eval/Images/")
except FileNotFoundError:
    print("The path doesn't exist.")
print("[INFO] Preparing the structer of the pictures folder.")
try:
    os.mkdir(r"./Duplicate")
except :
    pass

for dire in os.listdir(os.getcwd()):
    if (dire == "Duplicate"):
        continue
    try:
        os.mkdir(os.path.join(dire) + r"/Train/")
    except FileNotFoundError:
        print("\tThe directory couldn't be created because the directory doesn't exist.")
    except FileExistsError:
        print(f"\t{dire}-Train folder already created.")
    
    try:
        os.mkdir(os.path.join(dire) + r"/Evaluation/")
    except FileNotFoundError:
        print("\tThe directory couldn't be created because the directory doesn't exist.")
    except FileExistsError:
        print(f"\t{dire}-Evaluation folder already created.") 

### Image Hashing

In [ ]:
import os
import numpy as np
import cv2 as cv
import shutil

In [ ]:
#Klasse Graue Bilde erstellen un skallieren
def gray_file(img, hashSize = 8):
    #Gray-Scale the Image
    img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

    #Resize the grayscale image, adding a single column (width) so we can compute the horizontal gradient
    img_gray_resize = cv.resize(img_gray, (hashSize + 1, hashSize))

    #Compute the (relative) horizontal gradient between adjacent column pixels
    diff = img_gray_resize[:, 1:] > img_gray_resize[:, :-1]

    #Convert the difference image to a hash
    return sum([2 ** i for (i, v) in enumerate(diff.flatten()) if v])

In [ ]:
def convert_hash(h):
    return int(np.array(h, dtype="float64"))

In [ ]:
#Initialisierung
project_folder = r"D:/GitHub/GML_Project"
os.chdir(project_folder)
os.chdir(r"./Pictures/Tierbilder_Dataset_Train_Eval/Images")

In [ ]:
#Einlesen der Dateinamen und erstellen des Gray-Convert
hashes = {}
img=''

all_files = []
all_dirs = []
all_files_path = []

for dirs in os.listdir(os.getcwd()):
    all_dirs.append(dirs)
    for files in os.listdir(dirs):
        if (files != "Train" and files != "Evaluation"):
            all_files.append(files)
            all_files_path.append(os.path.join(dirs, files))

for i, files in enumerate(all_files_path):
    img = cv.imread(files)
    #Compute the hash for the Image and convert it
    the_hash = gray_file(img, 8)
    the_hash = convert_hash(the_hash)

    #update the hashes dictionary
    same_hash = hashes.get(the_hash, [])
    same_hash.append(os.path.join(files))
    hashes[the_hash] = same_hash

In [ ]:
#All image names with the same hash are in one list
all_multiple_hashes = []
for hasch in hashes:
    if (len(hashes[hasch]) > 1 ):
        all_multiple_hashes.append(hashes[hasch])

In [ ]:
for counter, hasch in enumerate(all_multiple_hashes):
    for i in range(len(all_multiple_hashes[counter])):
        if (i == 0):
            print(f"The duplicate image {counter +1} will be moved to the duplicate folder.")
            try:
                os.mkdir(os.path.join(r"./Duplicate", str(counter)))
            except:
                print(f"The folder for the {counter +1} Hash is already created.")
        else:
            #print(all_multiple_hashes[counter][i])
            try:
                shutil.move(os.path.join(os.getcwd(), all_multiple_hashes[counter][i]), os.path.join(r"./Duplicate", str(counter)))
            except:
                print("The duplicate file is already moved.")